In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression, f_classif
from matplotlib.colors import ListedColormap

In [ ]:
def load_dataset(df):
    data = df[[c for c in df if c not in ['DFT CD Intensity (a.u.)']] + ['DFT CD Intensity (a.u.)']]
    dataset = data.values
    X = dataset[:, :-1]
    y = dataset[:, -1].reshape(-1, 1)

    return X, y

In [ ]:
def select_features(X_train, y_train, X_test):
    fs = SelectKBest(score_func = f_regression, k = 'all')
    fs.fit(X_train, y_train.ravel())
    X_train_fs = fs.transform(X_train)
    X_test_fs = fs.transform(X_test)

    return X_train_fs, X_test_fs, fs

In [ ]:
def get_feature_scores(data, fs_scores_):
    feature_names = data.columns[:-1]
    feature_scores = [(feature_names[i], score) for i, score in enumerate(fs_scores_)]
    df_scores = pd.DataFrame(feature_scores, columns = ['Feature', 'Score'])
    df_scores = df_scores.sort_values(by = 'Score', ascending = False).reset_index(drop = True)
    return df_scores

In [ ]:
def process_df(df, feature_code):
    # feature_code: [old/new/no]-pol_[old/new/no]-pi_[y/n]-ecfp_[all/no/x/y/z]-torsion_[all/no/x/y/z]-sd
    if feature_code.endswith('_pt'):
        feature_code = feature_code.split('_pt')[0]
        first_cols = ['Monomer 1', 'Monomer 2']
    else:
        first_cols = ['Monomer 1', 'Monomer 2', 'Oligomer Type']
        
    pol, pi, ecfp = feature_code.split('_')[:-2]
    torsion_selection = feature_code.split('_')[-2]
    sd_selection = feature_code.split('_')[-1]
    
    geo_options = ['x', 'y', 'z']
    torsion_options = ['Total Torsion Angle', 'Mean Torsion Angle', 'Inner Mean Torsion Angle']
    sd_options = ['SD Torsions', 'SD Difference']
    
    geo_select_dict = {
        'all-torsion': [f"{t_opt}_{g_opt}" for t_opt in torsion_options for g_opt in geo_options],
        'all-sd': [f"{s_opt}_{g_opt}" for s_opt in sd_options for g_opt in geo_options],
        'no-torsion': [],
        'no-sd': [],
        'x-torsion': [f"{t_opt}_x" for t_opt in torsion_options],
        'y-torsion': [f"{t_opt}_y" for t_opt in torsion_options],
        'z-torsion': [f"{t_opt}_z" for t_opt in torsion_options],
        'x-sd': [f"{s_opt}_x" for s_opt in sd_options],
        'y-sd': [f"{s_opt}_y" for s_opt in sd_options],
        'z-sd': [f"{s_opt}_z" for s_opt in sd_options],
    }
    
    others_select_dict = {
        'old-pol': ['Molecule Polarizability', 'Average Monomer Polarizability', 'Dipole Moment'],
        'new-pol': ['Rotated Polarizability', 'Average Monomer Polarizability', 'Dipole Moment', 'α_xx', 'α_xy', 'α_yy'],
        'old-pi': ['PiSystemSize'],
        'new-pi': ['PiSystemSize', 'PiSystemArea', 'PiSystemLength'],
        'y-ecfp': [f"ECFP_{i}" for i in range(4096)],
        'no-pi': [],
        'no-pol': [],
        'n-ecfp': []
    }
    
    pol_cols = others_select_dict.get(pol)
    torsion_cols = geo_select_dict.get(torsion_selection)
    sd_cols = geo_select_dict.get(sd_selection)
    pi_cols = others_select_dict.get(pi)
    ecfp_cols = others_select_dict.get(ecfp)
    spec_cols = ['DFT Oscillator Strength', 'DFT CD Intensity']
    
    if pol.split('-')[0] != 'no' and pi.split('-')[0] != 'no':
        energy_cols = [
            'Molecule HOMO (eV)', 'Molecule LUMO (eV)', 'Molecule HLG (eV)', 
            'ΔE_HOMO (eV)', 'ΔE_LUMO (eV)', 'Monomer HLG (eV)'
        ]
        first_cols.extend(energy_cols)

    col_order = first_cols + pol_cols + torsion_cols + sd_cols + pi_cols + ecfp_cols + spec_cols
    
    df = df[col_order]
    
    df = df.rename(columns = {
        'DFT CD Intensity': 'DFT CD Intensity (a.u.)',
        'DFT Oscillator Strength': 'DFT Oscillator Strength (a.u.)',
    })
    
    new_col_names = []
    for col in df.columns:
        if 'Torsion' in col or 'SD' in col:
            new_col_names.append(col + ' (°)')
        elif 'Polarizability' in col or 'α' in col:
            new_col_names.append(col + ' (Å$^3$)')
        elif 'Dipole' in col:
            new_col_names.append(col + ' (Debye)')
        else:
            new_col_names.append(col)
    
    df.columns = new_col_names
    
    if len(torsion_selection) < 2:
        df = df.rename(columns = {
            'Total Torsion Angle_z (°)': 'Total Torsion Angle (°)',
            'Mean Torsion Angle_z (°)': 'Mean Torsion Angle (°)',
            'Inner Mean Torsion Angle_z (°)': 'Inner Mean Torsion Angle (°)',})
    if len(sd_selection) < 2:
        df = df.rename(columns = {
            'SD Torsions_x (°)': 'SD Torsions (°)',
            'SD Difference_x (°)': 'SD Difference (°)'
        })
        
    df = df.dropna().reset_index(drop = True)
    
    print(f'Number of features (m): {len(df.columns)}')
    print(f'Number of data points (n): {len(df)}')
    print(f"Input features: {list(dict.fromkeys([x.split('_')[0] for x in df.drop(['DFT CD Intensity (a.u.)'], axis = 1).columns]))}\n")
    return df

In [ ]:
def anova_graph(anova_df, ft_code, output_path, output_name, custom_cmap10, preview):
    anova_features = anova_df['Feature'].tolist()
    anova_coeffs = anova_df['Score'].tolist()
    indices = np.argsort(anova_coeffs)
    num_features = min(10, len(anova_coeffs)) 

    my_cmap = plt.get_cmap(name=custom_cmap10)
    fig, ax = plt.subplots(figsize=(10, 6))

    selected_indices = indices[-num_features:][::-1]

    selected_coeffs = [anova_coeffs[i] for i in selected_indices][::-1]
    selected_features = [anova_features[i] for i in selected_indices][::-1]
    
    if 'all' in ft_code:
        for i in range(len(selected_features)):
            if '_x (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_x (°)', '_0 (°)')
            elif '_y (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_y (°)', '_1 (°)')
            elif '_z (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_z (°)', '_2 (°)')
                
    colors = [my_cmap(i / num_features) for i in reversed(range(num_features))]

    ax.barh(range(num_features), selected_coeffs, color=colors, edgecolor = 'k', align='center')
    plt.yticks(range(num_features), selected_features, fontsize=16)
    plt.xlabel('Score', fontsize=20)
    plt.xticks(fontsize=16)
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f"f-ANOVA_bar_{output_name}.pdf"))
    #plt.savefig(os.path.join(output_path, f"f-ANOVA_bar_{output_name}.png"), dpi = 300)
    if preview == False:
        plt.close()


def anova_ecfpsum_graph(anova_df, ft_code, ecfp_sum, output_path, output_name, custom_cmap10, preview):
    anova_features = anova_df['Feature'].tolist()
    anova_coeffs = anova_df['Score'].tolist()
    indices = np.argsort(anova_coeffs)
    num_features = min(10, len(anova_coeffs)) 

    fig, ax = plt.subplots(figsize=(10, 6))

    selected_indices = indices[-num_features:][::-1]
    selected_coeffs = [anova_coeffs[i] for i in selected_indices][::-1]
    selected_features = [anova_features[i] for i in selected_indices][::-1]

    if 'all' in ft_code:
        for i in range(len(selected_features)):
            if '_x (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_x (°)', '_0 (°)')
            elif '_y (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_y (°)', '_1 (°)')
            elif '_z (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_z (°)', '_2 (°)')
                
    added_label = 'ECFP4 Bits Sum'
    added_color = '#291EAB'

    final_coeffs = [ecfp_sum] + selected_coeffs
    final_features = [added_label] + selected_features

    my_cmap = plt.get_cmap(name=custom_cmap10)
    colors = [added_color] + [my_cmap(i / num_features) for i in reversed(range(num_features))]

    ax.barh(range(num_features + 1), final_coeffs, color=colors, edgecolor = 'k', align='center')
    plt.yticks(range(num_features + 1), final_features, fontsize=16)
    plt.xlabel('Score', fontsize=20)
    plt.xticks(fontsize=16)
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f"f-ANOVA_bar_{output_name}_ecfpsum.pdf"))
    #plt.savefig(os.path.join(output_path, f"f-ANOVA_bar_{output_name}_ecfpsum.png"), dpi = 300)
    if preview == False:
        plt.close()

In [ ]:
def main(df, ft_code, color_palette, output_path):
    os.makedirs(output_path, exist_ok = True)
    
    output_name = ft_dict.get(ft_code)
    if output_name.endswith('_pt'):
        df = df[df['Oligomer Type'] == 2].reset_index(drop = True)
        
    data_df = df.copy(deep = True)
    data_df = process_df(data_df, ft_code)

    X, y = load_dataset(data_df)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 4937548)
    X_train_fs, X_test_fs, fs = select_features(X_train, y_train, X_test)

    feature_score_df = get_feature_scores(data_df, fs.scores_)
    feature_score_df = feature_score_df[feature_score_df['Score'] > 0]
    feature_score_df.to_csv(os.path.join(output_path, f"f-ANOVA_{output_name}.csv"), index = False)
    top_feature_score_df = feature_score_df[~feature_score_df['Feature'].str.contains('ECFP')].reset_index(drop = True).head(10)
    anova_df = top_feature_score_df.copy()
    anova_graph(anova_df, ft_code, output_path, output_name, custom_cmap10, preview=False)

    ecfp_sum = 0

    for item in sorted(zip(feature_score_df['Feature'], feature_score_df['Score'])):
        if 'ECFP' in item[0]:
            ecfp_sum += item[1]

    ecfp_sum_df = pd.DataFrame({
        'Feature': ['ECFP Sum'],
        'Score': [ecfp_sum]
    })

    ecfp_feature_score_df = pd.concat([top_feature_score_df, ecfp_sum_df], ignore_index = True)
    ecfp_feature_score_df.to_csv(os.path.join(output_path, f"f-ANOVA_{output_name}_ecfp-sum.csv"), index = False)
    anova_ecfpsum_graph(anova_df, ft_code, ecfp_sum, output_path, output_name, custom_cmap10, preview=False) 


In [ ]:
ft_dict = {
    'old-pol_old-pi_y-ecfp_all-torsion_all-sd': 'ooyaa',
    'old-pol_old-pi_y-ecfp_z-torsion_x-sd': 'ooyzx',
    'old-pol_old-pi_y-ecfp_z-torsion_x-sd_pt': 'ooyzx_pt',
    'new-pol_old-pi_y-ecfp_z-torsion_x-sd': 'noyzx',
    'new-pol_new-pi_y-ecfp_z-torsion_x-sd': 'nnyzx',
}

ft_codes = list(ft_dict.keys())

color_palette_10 = ['#24E4BC', '#00CFF9', '#648FFF', '#785EF0', '#AA42B8', '#DC267F', '#ED2F2C', '#FE6100', '#FF8900', '#FFB000', '#FFE200']
custom_cmap10 = ListedColormap(color_palette_10, name = 'cb10')

df = pd.read_csv('clean_dft_df_4096.csv')
output_path = os.path.join(os.getcwd(), 'output')

In [ ]:
for i in range(len(ft_codes)):
    ft_code = ft_codes[i]
    main(df, ft_code, custom_cmap10, output_path)